In [1]:
import mne
import neurokit2 as nk
import pandas as pd
import numpy as np
import glob
import os

# --- AYARLAR ---
# Verilerin olduğu ana klasör (r harfi Windows yollarındaki \ işaretini düzeltir)
root_path = r"C:\Users\Zeynep Şevval\Desktop\TEZ\00FinalAnaliz\Fizyolojik\Raw"

# Hedeflenen yeni örnekleme hızı (İşlem hızı için 50Hz idealdir)
target_sampling_rate = 50

# Sonuçları depolayacağımız liste
all_results = []

# --- 1. DOSYALARI BULMA ---
# Alt klasörler dahil (recursive=True) tüm .vhdr dosyalarını bulur
file_list = glob.glob(os.path.join(root_path, '**', '*.vhdr'), recursive=True)

print(f"Toplam {len(file_list)} adet veri dosyası bulundu. İşlem başlıyor...")

# --- 2. DÖNGÜ (HER KATILIMCI İÇİN) ---
for file_path in file_list:
    try:
        # Dosya adını al (örn: "Katilimci_01.vhdr") - Tabloya yazdırmak için
        filename = os.path.basename(file_path)
        print(f"İşleniyor: {filename}...")

        # MNE ile veriyi oku (preload=True veriyi hafızaya alır)
        raw = mne.io.read_raw_brainvision(file_path, preload=True, verbose=False)
        
        # Orijinal örnekleme hızını al
        original_sf = raw.info['sfreq']
        
        # Veriyi DataFrame'e çevir
        df = raw.to_data_frame()
        
        # ---------------------------------------------------------
        # BÖLÜM A: GSR ANALİZİ (cvxEDA Yöntemi - Hernando-Gallego et al.)
        # ---------------------------------------------------------
        if 'GSR' in df.columns:
            gsr_signal = df['GSR']
            
            # 1. Downsample (Hızlandırmak için)
            gsr_down = nk.signal_resample(gsr_signal, sampling_rate=original_sf, desired_sampling_rate=target_sampling_rate)
            
            # 2. Filtrele (5Hz Low-pass) - Gürültü temizliği
            gsr_clean = nk.signal_filter(gsr_down, sampling_rate=target_sampling_rate, highcut=5, method='butterworth', order=4)
            
            # 3. Ayrıştır (Decomposition) - Phasic ve Tonic
            # cvxEDA yöntemi tez için en bilimsel olanıdır
            gsr_decomposed = nk.eda_phasic(nk.standardize(gsr_clean), sampling_rate=target_sampling_rate, method='cvxeda')
            
            # 4. Tezin için gerekli sayıları çıkar
            scr_peaks, info = nk.eda_peaks(gsr_decomposed['EDA_Phasic'].values, sampling_rate=target_sampling_rate)
            
            n_scr = len(info['SCR_Peaks']) # Toplam SCR tepki sayısı
            mean_tonic = np.mean(gsr_decomposed['EDA_Tonic']) # Ortalama SCL seviyesi
            mean_phasic = np.mean(gsr_decomposed['EDA_Phasic']) # Ortalama tepki şiddeti
            
        else:
            print(f"UYARI: {filename} dosyasında 'GSR' kanalı bulunamadı.")
            n_scr, mean_tonic, mean_phasic = (np.nan, np.nan, np.nan)

        # ---------------------------------------------------------
        # BÖLÜM B: BP (BLOOD PULSE) ANALİZİ (Elgendi Yöntemi)
        # ---------------------------------------------------------
        if 'BP' in df.columns:
            ppg_signal = df['BP']
            
            # 1. Downsample
            ppg_down = nk.signal_resample(ppg_signal, sampling_rate=original_sf, desired_sampling_rate=target_sampling_rate)
            
            # 2. Filtrele (0.5 - 8 Hz Bandpass) - Nefes ve kas gürültüsünü at
            ppg_clean = nk.signal_filter(ppg_down, sampling_rate=target_sampling_rate, lowcut=0.5, highcut=8, method='butterworth', order=4)
            
            # 3. Analiz Et (NeuroKit2 otomatik analiz fonksiyonu)
            # Bu fonksiyon Nabız, RMSSD, pNN50 hepsini hesaplar
            ppg_results, _ = nk.ppg_process(ppg_clean, sampling_rate=target_sampling_rate)
            ppg_analyze = nk.ppg_analyze(ppg_results, sampling_rate=target_sampling_rate)
            
            mean_hr = ppg_analyze['PPG_Rate_Mean'].values[0] # Ortalama Nabız
            rmssd = ppg_analyze['HRV_RMSSD'].values[0]       # HRV (Duygusal regülasyon)
            pnn50 = ppg_analyze['HRV_pNN50'].values[0]       # Parasempatik aktivite
            
        else:
            print(f"UYARI: {filename} dosyasında 'BP' kanalı bulunamadı.")
            mean_hr, rmssd, pnn50 = (np.nan, np.nan, np.nan)

        # ---------------------------------------------------------
        # SONUÇLARI KAYDET
        # ---------------------------------------------------------
        all_results.append({
            'Dosya_Adi': filename,
            'GSR_SCR_Sayisi': n_scr,
            'GSR_Ortalama_Tonic_SCL': mean_tonic,
            'GSR_Ortalama_Phasic': mean_phasic,
            'BP_Ortalama_Nabiz': mean_hr,
            'BP_HRV_RMSSD': rmssd,
            'BP_HRV_pNN50': pnn50
        })

    except Exception as e:
        print(f"HATA: {filename} işlenirken bir sorun oluştu: {e}")

# --- 3. CSV OLUŞTURMA ---
if all_results:
    results_df = pd.DataFrame(all_results)
    save_path = os.path.join(root_path, "TEZ_SONUCLARI.csv")
    results_df.to_csv(save_path, index=False)
    print(f"\nİŞLEM BAŞARIYLA TAMAMLANDI! Sonuçlar şurada: {save_path}")
    print(results_df.head()) # İlk 5 satırı göster
else:
    print("\nHiçbir sonuç üretilemedi.")

Toplam 275 adet veri dosyası bulundu. İşlem başlıyor...
İşleniyor: fp18_fp_gk.vhdr...
İşleniyor: p10_fn.vhdr...
İşleniyor: p10_fn_ga.vhdr...
İşleniyor: p10_fn_gk.vhdr...
İşleniyor: p10_fp.vhdr...
İşleniyor: p10_fp_ga.vhdr...
İşleniyor: p10_fp_gk.vhdr...
İşleniyor: p10_ga.vhdr...
İşleniyor: p10_gk.vhdr...
İşleniyor: p11_fn.vhdr...
İşleniyor: p11_fn_ga.vhdr...
İşleniyor: p11_fn_gk.vhdr...
İşleniyor: p11_fp.vhdr...
İşleniyor: p11_fp_ga.vhdr...
İşleniyor: p11_fp_gk.vhdr...
İşleniyor: p11_ga.vhdr...
İşleniyor: p11_gk.vhdr...
İşleniyor: p12_fn.vhdr...
İşleniyor: p12_fn_ga.vhdr...
İşleniyor: p12_fn_gk.vhdr...
İşleniyor: p12_fp.vhdr...
İşleniyor: p12_fp_ga.vhdr...
İşleniyor: p12_fp_gk.vhdr...
İşleniyor: p12_ga.vhdr...
İşleniyor: p12_gk.vhdr...
İşleniyor: p13_fn.vhdr...
İşleniyor: p13_fn_ga.vhdr...
İşleniyor: p13_fn_gk.vhdr...
İşleniyor: p13_fp.vhdr...
İşleniyor: p13_fp_ga.vhdr...
İşleniyor: p13_fp_gk.vhdr...
İşleniyor: p13_ga.vhdr...
İşleniyor: p13_gk.vhdr...
İşleniyor: p14_fn.vhdr...
İşleniyo

c:\Users\Zeynep Şevval\AppData\Local\Programs\Python\Python313\Lib\site-packages\neurokit2\hrv\hrv_nonlinear.py:536: NeuroKitWarning: DFA_alpha2 related indices will not be calculated. The maximum duration of the windows provided for the long-term correlation is smaller than the minimum duration of windows. Refer to the `scale` argument in `nk.fractal_dfa()` for more information.
  warn(


İşleniyor: p27_fn_gk.vhdr...
İşleniyor: p27_fp.vhdr...
İşleniyor: p27_fp_ga.vhdr...


c:\Users\Zeynep Şevval\AppData\Local\Programs\Python\Python313\Lib\site-packages\neurokit2\hrv\hrv_nonlinear.py:536: NeuroKitWarning: DFA_alpha2 related indices will not be calculated. The maximum duration of the windows provided for the long-term correlation is smaller than the minimum duration of windows. Refer to the `scale` argument in `nk.fractal_dfa()` for more information.
  warn(


İşleniyor: p27_fp_gk.vhdr...
İşleniyor: p27_ga.vhdr...
İşleniyor: p27_gk.vhdr...
İşleniyor: p28_fn.vhdr...
İşleniyor: p28_fn_ga.vhdr...
İşleniyor: p28_fn_gk.vhdr...
İşleniyor: p28_fp.vhdr...
İşleniyor: p28_fp_ga.vhdr...
İşleniyor: p28_fp_gk.vhdr...
İşleniyor: p28_ga.vhdr...
İşleniyor: p28_gk.vhdr...
İşleniyor: p29_fn.vhdr...
İşleniyor: p29_fn_ga.vhdr...
İşleniyor: p29_fn_gk.vhdr...
İşleniyor: p29_fp.vhdr...
İşleniyor: p29_fp_ga.vhdr...
İşleniyor: p29_fp_gk.vhdr...
İşleniyor: p29_ga.vhdr...
İşleniyor: p29_gk.vhdr...
İşleniyor: p30.vhdr...
İşleniyor: p30_fn.vhdr...
İşleniyor: p30_fn_ga.vhdr...
İşleniyor: p30_fn_gk.vhdr...
İşleniyor: p30_fp.vhdr...
İşleniyor: p30_fp_ga.vhdr...
İşleniyor: p30_fp_gk.vhdr...
İşleniyor: p30_gk.vhdr...
İşleniyor: p31_fn.vhdr...
İşleniyor: p31_fn_ga.vhdr...
İşleniyor: p31_fn_gk.vhdr...
İşleniyor: p31_fp.vhdr...
İşleniyor: p31_fp_ga.vhdr...
İşleniyor: p31_fp_gk.vhdr...
İşleniyor: p31_ga.vhdr...
İşleniyor: p31_gk.vhdr...
İşleniyor: p32_fn.vhdr...
İşleniyor: p32_f

...

İşleniyor: p8_ga.vhdr...


c:\Users\Zeynep Şevval\AppData\Local\Programs\Python\Python313\Lib\site-packages\neurokit2\hrv\hrv_nonlinear.py:536: NeuroKitWarning: DFA_alpha2 related indices will not be calculated. The maximum duration of the windows provided for the long-term correlation is smaller than the minimum duration of windows. Refer to the `scale` argument in `nk.fractal_dfa()` for more information.
  warn(


İşleniyor: p8_gk.vhdr...


c:\Users\Zeynep Şevval\AppData\Local\Programs\Python\Python313\Lib\site-packages\neurokit2\hrv\hrv_nonlinear.py:536: NeuroKitWarning: DFA_alpha2 related indices will not be calculated. The maximum duration of the windows provided for the long-term correlation is smaller than the minimum duration of windows. Refer to the `scale` argument in `nk.fractal_dfa()` for more information.
  warn(


İşleniyor: p9_fn.vhdr...
İşleniyor: p9_fn_ga.vhdr...
İşleniyor: p9_fn_gk.vhdr...
İşleniyor: p9_fp.vhdr...
İşleniyor: p9_fp_ga.vhdr...
İşleniyor: p9_fp_gk.vhdr...
İşleniyor: p9_ga.vhdr...
İşleniyor: p9_gk.vhdr...
İşleniyor: pl2fpgk.vhdr...
İşleniyor: pl2_fn.vhdr...
İşleniyor: pl2_fn_ga.vhdr...
İşleniyor: pl2_fn_gk.vhdr...
İşleniyor: pl2_fp.vhdr...
İşleniyor: pl2_fp_ga.vhdr...
İşleniyor: pl2_ga1.vhdr...
İşleniyor: pl2_gk1.vhdr...
İşleniyor: pl3_fn.vhdr...
İşleniyor: pl3_fn_ga.vhdr...
İşleniyor: pl3_fn_gk.vhdr...
İşleniyor: pl3_fp.vhdr...
İşleniyor: pl3_fp_ga.vhdr...
İşleniyor: pl3_fp_gk.vhdr...
İşleniyor: PL3_ga1.vhdr...
İşleniyor: pl3_gk1.vhdr...
İşleniyor: pl4_ga.vhdr...
İşleniyor: pl4_gk.vhdr...
İşleniyor: pl5_ga.vhdr...
İşleniyor: pl6_fn.vhdr...
İşleniyor: pl6_fn_ga.vhdr...
İşleniyor: pl6_fn_gk.vhdr...
İşleniyor: pl6_fp.vhdr...
İşleniyor: pl6_fp_ga.vhdr...
İşleniyor: pl6_fp_gk.vhdr...
İşleniyor: pl6_ga.vhdr...
İşleniyor: pl6_gk.vhdr...
İşleniyor: PL7_ga.vhdr...

İŞLEM BAŞARIYLA TAMAM